# Radar Tech Brasil - Análise Exploratória

Este notebook apresenta uma primeira leitura exploratória do mercado formal de tecnologia no Brasil a partir do Novo CAGED e da CBO.

A análise usa agregados processados pelo pipeline do projeto, evitando carregar os microdados completos em memória.

## 1. Contexto

O objetivo é transformar microdados públicos em indicadores analíticos sobre admissões, desligamentos, saldo, ocupações, categorias de tecnologia, distribuição geográfica, remuneração e perfil profissional.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

def read_agg(filename: str) -> pd.DataFrame:
    return pd.read_csv(PROCESSED_DIR / filename, sep=';')

## 2. Fonte dos Dados

- Novo CAGED: microdados públicos do Ministério do Trabalho e Emprego.
- CBO: Classificação Brasileira de Ocupações usada para mapear ocupações de tecnologia.
- Janela inicial: competências `202507` a `202606`.

A metodologia de classificação CBO tech está documentada em `docs/metodologia_cbo_tech.md`.

In [ ]:
overview = read_agg('agg_tech_overview_mensal.csv')
category = read_agg('agg_tech_by_category_mensal.csv')
occupation = read_agg('agg_tech_by_occupation_mensal.csv')
uf = read_agg('agg_tech_by_uf_mensal_enriched.csv')
age = read_agg('agg_tech_by_age_group_mensal.csv')
education = read_agg('agg_tech_by_education_mensal_enriched.csv')

overview['competencia'] = overview['competencia'].astype(str)
category['competencia'] = category['competencia'].astype(str)
occupation['competencia'] = occupation['competencia'].astype(str)
uf['competencia'] = uf['competencia'].astype(str)
age['competencia'] = age['competencia'].astype(str)
education['competencia'] = education['competencia'].astype(str)

overview.head()

## 3. Qualidade dos Dados

O pipeline preserva registros e cria flags de qualidade em vez de remover dados automaticamente. Para remuneração média e mediana, foram considerados apenas salários maiores que zero e sem flag de salário extremo.

In [ ]:
overview.describe(include='all')

## 4. Mercado Geral

A primeira leitura observa volume total, admissões, desligamentos e saldo mensal do recorte tech.

In [ ]:
total_admissoes = int(overview['total_admissoes'].sum())
total_desligamentos = int(overview['total_desligamentos'].sum())
saldo = int(overview['saldo_empregos'].sum())

pd.DataFrame([
    {'metrica': 'Admissões tech', 'valor': total_admissoes},
    {'metrica': 'Desligamentos tech', 'valor': total_desligamentos},
    {'metrica': 'Saldo tech', 'valor': saldo},
])

## 5. Evolução Temporal

A série mensal evita resumir a janela em um único número e mostra meses com comportamento distinto.

In [ ]:
px.line(
    overview,
    x='competencia',
    y=['total_admissoes', 'total_desligamentos', 'saldo_empregos'],
    markers=True,
    labels={'value': 'Registros', 'competencia': 'Competência', 'variable': 'Métrica'},
    title='Evolução mensal de admissões, desligamentos e saldo tech',
)

## 6. Categorias de Tecnologia

Categorias ajudam a separar ocupações de desenvolvimento, suporte, redes, infraestrutura, dados, segurança e gestão.

In [ ]:
category_total = (
    category.groupby('categoria_tech', as_index=False)
    .agg(registros=('registros', 'sum'), saldo_empregos=('saldo_empregos', 'sum'))
    .sort_values('registros', ascending=False)
)

px.bar(
    category_total,
    x='registros',
    y='categoria_tech',
    orientation='h',
    labels={'registros': 'Registros', 'categoria_tech': 'Categoria'},
    title='Volume por categoria tech',
)

## 7. Ocupações

O ranking de ocupações mostra onde está concentrado o volume de movimentações formais no recorte CBO tech.

In [ ]:
occupation_total = (
    occupation.groupby(['codigo_cbo', 'ocupacao', 'categoria_tech'], as_index=False)
    .agg(registros=('registros', 'sum'), saldo_empregos=('saldo_empregos', 'sum'))
    .sort_values('registros', ascending=False)
    .head(15)
)

occupation_total

## 8. Estados

A análise geográfica usa UF enriquecida com sigla, nome e região.

In [ ]:
uf_total = (
    uf.groupby(['uf', 'uf_sigla', 'uf_nome', 'regiao_nome'], as_index=False)
    .agg(registros=('registros', 'sum'), saldo_empregos=('saldo_empregos', 'sum'))
    .sort_values('registros', ascending=False)
)

px.bar(
    uf_total.head(20),
    x='uf_sigla',
    y='saldo_empregos',
    labels={'uf_sigla': 'UF', 'saldo_empregos': 'Saldo'},
    title='Saldo tech por UF',
)

## 9. Salários

A remuneração é analisada com a regra documentada de salários válidos. A mediana é especialmente útil porque reduz a influência de valores extremos.

In [ ]:
px.line(
    overview,
    x='competencia',
    y=['remuneracao_media', 'remuneracao_mediana'],
    markers=True,
    labels={'value': 'Remuneração', 'competencia': 'Competência', 'variable': 'Métrica'},
    title='Evolução de remuneração média e mediana',
)

## 10. Perfil Profissional

Faixa etária e escolaridade ajudam a entender o perfil dos vínculos formais no recorte tech.

In [ ]:
age_total = (
    age.groupby('faixa_etaria', as_index=False)
    .agg(registros=('registros', 'sum'), saldo_empregos=('saldo_empregos', 'sum'))
)
age_order = ['Ate 20', '21-25', '26-30', '31-35', '36-40', '41-50', '51+', 'Nao informado']
age_total['ordem'] = age_total['faixa_etaria'].apply(lambda value: age_order.index(value) if value in age_order else 99)
age_total = age_total.sort_values('ordem').drop(columns='ordem')

px.bar(age_total, x='faixa_etaria', y='registros', title='Registros por faixa etária')

In [ ]:
education_total = (
    education.groupby(['grau_instrucao', 'escolaridade'], as_index=False)
    .agg(registros=('registros', 'sum'), saldo_empregos=('saldo_empregos', 'sum'))
    .sort_values('grau_instrucao')
)

education_total

## 11. Principais Insights

Os insights gerados automaticamente estão documentados em `docs/insights_iniciais.md`.

Eles devem ser lidos como evidências descritivas, não como inferências causais.

## 12. Limitações

- O recorte depende do mapeamento CBO tech `v0.1`.
- CBO não captura senioridade, stack, modalidade remota ou tipo de contrato em detalhe moderno.
- A janela inicial possui 12 competências.
- RAIS, IBGE e Banco Central ainda não foram integrados.

## 13. Próximas Análises

- Salário por categoria e UF.
- Sazonalidade mensal.
- Comparação entre regiões.
- Revisão do mapeamento CBO tech com novas fontes e validação qualitativa.
- Carga PostgreSQL detalhada para consultas mais flexíveis.